習題一：使用random模組，產生出一個介於0到1之間的小數$r$，使用while迴圈，計算出$r$的前二十位二進位表示$r = 0.b_1b_2b_3\cdots {b_{20}}_2$。輸出格式如附圖Exercise20261015.png

import random

# 1. 產生一個介於 0 到 1 之間的小數 r
r = random.random()

# 2. 準備用來儲存二進位結果的字串
binary_digits = []

# 3. 使用 while 迴圈計算前二十位二進位表示
count = 0
temp_r = r

while count < 20:
  temp_r *= 2
  bit = int(temp_r)
  binary_digits.append(str(bit))
  temp_r -= bit
  count += 1

# 4. 依照附圖的指定格式輸出結果
binary_str = "".join(binary_digits)
print(f"假如 r = {r}")
print(f"r ≈ 0.{binary_str}_2")

Python 的 `int` 型別採用 Arbitrary-Precision Integer（任意精度整數）設計，能夠精確表示任意大小的整數，其實作方式可參考 CPython 的 `PyLongObject`。與 C/C++ 的 `int` 或 `long long` 不同，Python 的整數不受固定 32 位元或64 位元限制，而是會依據數值大小自動配置更多記憶體，因此只要記憶體足夠，便能表示極大的整數。例如：
```python
n = 10**68
print(n)
```
上述程式可正常執行，表示 Python 能夠精確儲存$10^{68}$，即中文數字中的「無量大數」。


可利用下列程式觀察不同大小整數所需的位元數（bit）與記憶體空間：

```python
import sys
n = 10**68
print(f"bit_length = {n.bit_length()}")
print(f"memory = {sys.getsizeof(n)} bytes")
```

並查閱官方文件，了解：

- `bit_length()`
- `sys.getsizeof()`
- `PyLongObject`

三者之間的關係。

習題二：輸入一個正整數 \(n\)，其中$1 \le n \le 10^{68}$，請將 \(n\) 轉換為中文大寫數字表示法。需支援下列數字單位：萬、億、兆、京、垓、秭、穰、溝、澗、正、載、極、恆河沙、阿僧祇、那由他、不可思議、無量大數。比如輸入31415926，輸出參仟壹佰肆拾壹萬伍仟玖佰貳拾陸。

提示：思考利用字典，建立「數字」與「中文大寫數字」的對應關係。

def int_to_chinese_big_num(n_str):
  # 1. 定義中文數字與基礎單位
  num_chars = ["零", "壹", "貳", "參", "肆", "伍", "陸", "柒", "捌", "玖"]
  pos_units = ["", "拾", "佰", "仟"]

  # 2. 定義大單位（每 4 位為一個大節，對應題目要求的完整單位序列）
  # 預設對應：個/萬/億/兆/京/垓/秭/穰/溝/澗/正/載/極/恆河沙/阿僧祇/那由他/不可思議/無量大數
  big_units = [
      "",
      "萬",
      "億",
      "兆",
      "京",
      "垓",
      "秭",
      "穰",
      "溝",
      "澗",
      "正",
      "載",
      "極",
      "恆河沙",
      "阿僧祇",
      "那由他",
      "不可思議",
      "無量大數",
  ]

  # 移除開頭可能多餘的零或空白
  n_str = str(n_str).lstrip("0")
  if not n_str:
    return "零"

  # 3. 將字串從右到左每 4 位切成一組
  length = len(n_str)
  groups = []
  for i in range(length, 0, -4):
    start = max(0, i - 4)
    groups.append(n_str[start:i])

  res_parts = []
  zero_flag = False  # 標記是否需要補「零」

  # 4. 從最高的大單位區塊開始處理
  for g_idx, group in enumerate(reversed(groups)):
    # 實際大單位的索引
    actual_big_idx = len(groups) - 1 - g_idx
    if actual_big_idx >= len(big_units):
      break  # 超出定義時防呆

    group_int = int(group)
    if group_int == 0:
      # 如果整個 4 位區塊都是 0，保留零標記但不印出單位
      zero_flag = True
      continue

    # 轉換當前 4 位數區塊內部
    part_str = ""
    g_len = len(group)
    for i, ch in enumerate(group):
      digit = int(ch)
      power = g_len - 1 - i  # 該位數的小單位 (仟, 佰, 拾, "")

      if digit != 0:
        if zero_flag:
          part_str += "零"
          zero_flag = False
        part_str += num_chars[digit] + pos_units[power]
      else:
        # 遇到 0 的處理：如果後面還有非零數字，則標記補零
        if not part_str.endswith("零") and i < g_len - 1:
          zero_flag = True

    # 組合該區塊的大單位（例如：萬、億等）
    part_str += big_units[actual_big_idx]
    res_parts.append(part_str)

  return "".join(res_parts)


# --- 測試範例 ---
if __name__ == "__main__":
  # 測試題目中的範例：31415926
  test_n = 31415926
  print(f"輸入：{test_n}")
  print(f"輸出：{int_to_chinese_big_num(test_n)}")

習題三：請勿使用for、range，而是利用iterable、iter()、next()、while、StopIteration來設計九九乘法表。

# 1. 建立被乘數的 iterable（數字 1 到 9）
i_list = [1, 2, 3, 4, 5, 6, 7, 8, 9]

# 2. 使用 while 迴圈與疊代器印出九九乘法表
i_iter = iter(i_list)

while True:
  try:
    i = next(i_iter)

    # 3. 針對每個 i，內部再建立一個乘數的疊代器 (1 到 9)
    j_iter = iter([1, 2, 3, 4, 5, 6, 7, 8, 9])

    while True:
      try:
        j = next(j_iter)
        # 格式化輸出乘法算式與結果
        print(f"{i} x {j} = {i * j:2d}", end="   ")
      except StopIteration:
        # 當內部 j 走訪完畢，換行並跳出內部 while
        print()
        break

  except StopIteration:
    # 當外部 i 走訪完畢，結束整個外層 while
    break